# 14 · tmQMg structures

rxembed does not ship the [tmQMg](https://github.com/hkneiding/tmQMg) dataset. Clone it and point
`TMQMG_DIR` below at its `data/` directory. Reading these files needs xyzgraph for connectivity
perception: install the `workflow` extra, `pip install 'rxembed[workflow]'`.

In [ ]:
import csv
from pathlib import Path

from rdkit import RDLogger

import rxembed as rx

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

TMQMG_DIR = Path("path/to/tmQMg/data")  # a local clone's data/ directory

CHARGES = {
    row["id"]: int(row["charge"])
    for row in csv.DictReader((TMQMG_DIR / "tmQMg_properties_and_targets.csv").open())
}

## One structure by refcode

EABHCP is a small octahedral platinum complex: two hydride, three bromide and one carbonyl donor.
`rx.metal` enumerates its coordination isomers without being told the input's own seating in advance.

In [ ]:
refcode = "EABHCP"
mol = rx.read_xyz(str(TMQMG_DIR / "xyz" / f"{refcode}.xyz"), charge=CHARGES[refcode], bond_orders="xyz2mol")
reference = rx.cxsmiles(mol)
print(f"{refcode}: {mol.GetNumAtoms()} atoms, charge {CHARGES[refcode]}")
print("input arrangement:", reference)

isomers = rx.metal(mol)
isomers.summary()
arrangements = {rx.cxsmiles(iso) for iso in isomers}
print(f"\n{len(isomers)} coordination isomers; input arrangement present: {reference in arrangements}")

## Embed

Pick the isomer that matches the input's own arrangement and embed it.

In [ ]:
iso = next(iso for iso in isomers if rx.cxsmiles(iso) == reference)
ensemble = rx.embed(iso, n=1, seed=1)
print("embedded arrangement matches the input:", rx.cxsmiles(ensemble.mol) == reference)

## A handful more

The same read, enumerate, embed path across a few more refcodes, chosen only to run quickly.

In [ ]:
refcodes = ["CHGACA", "HGCETS", "TEVSIW", "COBBAZ", "CUHXEJ"]

for code in refcodes:
    try:
        m = rx.read_xyz(str(TMQMG_DIR / "xyz" / f"{code}.xyz"), charge=CHARGES[code], bond_orders="xyz2mol")
        ref = rx.cxsmiles(m)
        isos = rx.metal(m)
        match = next((i for i in isos if rx.cxsmiles(i) == ref), None)
        if match is None:
            print(f"{code}: FAIL (input arrangement not enumerated)")
            continue
        ens = rx.embed(match, n=1, seed=1)
        print(f"{code}: PASS")
    except Exception as exc:
        print(f"{code}: FAIL ({exc})")